# 02 · Preprocesamiento
**Responsable:** Santiago · **Módulo 2**

**Objetivo.** Producir la matriz que usará Juan para el clustering: limpia, transformada, estandarizada y **sin género**.

Pasos: limpieza → transformación log → estandarización → muestra estratificada → verificación → guardado.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import preprocessing as pp
from src import viz

viz.set_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Limpieza

In [2]:
raw = pp.load_raw()
df, log = pp.clean(raw)
log

,paso,filas_antes,filas_despues,descartadas
0,nulos en variables/identificadores,114000,114000,0
1,tempo = 0 o duración = 0,114000,113842,158
2,duplicados por track_id,113842,89583,24259
3,duplicados por artista + nombre,89583,81198,8385
4,TOTAL,114000,81198,32802


## 2. Transformación y estandarización
`StandardScaler` deja cada variable con media 0 y desviación 1. Es obligatorio: `loudness` está en decibelios negativos
y `duration_ms` en cientos de miles; sin escalar, esas dos dominarían cualquier distancia euclídea y PCA solo
"vería" la variable con mayor varianza.

In [3]:
X = pp.transform_features(df)
Xs, scaler = pp.scale(X)
Xs.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
danceability,0.000,1.000,-2.887,2.409
energy,0.000,1.000,-2.467,1.412
loudness,-0.000,1.000,-7.228,2.491
speechiness,0.000,1.000,-0.659,6.635
acousticness,0.000,1.000,-0.969,1.962
instrumentalness,-0.000,1.000,-0.571,2.319
liveness,-0.000,1.000,-1.244,3.536
valence,-0.000,1.000,-1.765,2.019
tempo,-0.000,1.000,-3.102,4.073
duration_ms,0.000,1.000,-6.462,7.944


## 3. Muestra estratificada para clustering jerárquico
K-Means corre con todo el dataset, pero el jerárquico aglomerativo necesita memoria proporcional a n² y no escala a
cientos de miles de filas. Se marca una muestra proporcional por género (`in_sample = True`) para que **ambos métodos
puedan compararse sobre las mismas canciones** si hace falta.

In [4]:
in_sample = pp.stratified_sample(df)
print(f"Muestra: {int(in_sample.sum()):,} de {len(df):,} canciones")

comparacion = pd.DataFrame({
    "pct_total": df[cfg.GENRE_COL].value_counts(normalize=True),
    "pct_muestra": df.loc[in_sample, cfg.GENRE_COL].value_counts(normalize=True),
}).fillna(0)
print(f"Diferencia máxima de proporción por género: {(comparacion.pct_total - comparacion.pct_muestra).abs().max():.4f}")

Muestra: 19,997 de 81,198 canciones
Diferencia máxima de proporción por género: 0.0000


## 4. Verificaciones antes de guardar

In [5]:
# 1) El género NO está en la matriz que se usará para clusterizar
assert cfg.GENRE_COL not in Xs.columns and "track_genre" not in Xs.columns
# 2) Sin nulos ni infinitos
assert not Xs.isna().any().any() and np.isfinite(Xs.values).all()
# 3) Media ~0 y desviación ~1
assert np.allclose(Xs.mean(), 0, atol=1e-8) and np.allclose(Xs.std(ddof=0), 1, atol=1e-8)
# 4) ids únicos
assert df[cfg.ID_COL].is_unique
print("Todas las verificaciones pasaron")

Todas las verificaciones pasaron


## 5. Guardado de entregables

In [6]:
pp.save_preprocessed(df, Xs, scaler, in_sample)
log.to_csv(cfg.TABLES / "bitacora_limpieza.csv", index=False)
sorted(p.name for p in cfg.DATA_PROCESSED.iterdir())

['.gitkeep',
 'X_pca.csv',
 'X_scaled.csv',
 'config_preprocesamiento.json',
 'pca.joblib',
 'scaler.joblib',
 'songs_clean.csv']

## 6. Decisiones a documentar en el informe
> **Completar tras ejecutar:** cuántas filas se descartaron en cada paso (tabla del punto 1), por qué se conservó solo
> la primera aparición de cada canción multigénero (`n_genres` queda guardado por si se quiere analizar) y qué variables
> se transformaron con log.